# CS604 Colab launcher — Ana: what causes the errors, on all 10,000 classes? (no training)

The same cause breakdown as `colab_launcher_ana_error_causes.ipynb`, but on the **all-class holdout** LoRA checkpoint (`lora_20261026_123804`: LoRA r24 no extras, 10,000 classes, 40 fit + 5 holdout photos per class, 144 training minutes, 77.78% top-1) instead of the 1,000-class pilot. The pilot under-counts siblings: its 1,000 random classes leave out most classes' close relatives.

| Cause | Detected when |
|---|---|
| sibling | the wrong guess is within 5 class IDs or among the true class's 5 most similar classes |
| unseen_look | the holdout photo is unlike every training photo of its class (e.g. a leafy tree when the training photos show bare trees) |
| two_looks | the class's training photos split into two groups and the holdout matches the smaller one (e.g. adult moth vs mostly caterpillars) |
| messy_class | at least 20% of the class's training photos look more like another class (mislabels, off-topic photos) |

Causes can overlap. Each share is compared with the same flag among **correctly** classified photos: a cause matters if errors have it much more often.

**Before running:** the run folder must be on Drive at `CS604_RUNS/ablation_allclass_holdout_lora_no_extras/runs/A_all_classes_144min/lora_20261026_123804/` (with `last.pt` and `config.yaml`). Step 3 checks it.

Outputs in `CS604_RUNS/allclass_holdout_analysis/causes/`: `causes.json`, `causes_per_error.csv`, `classes.csv`, `causes_<cause>.png`. About 45 min on an A100: ~25 min data download, ~10–15 min embedding 450,000 photos, a few minutes of analysis. Use **Runtime → Run all**.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
PILOT_DATA_DIR = "/content/pilot_data"
# The all-class holdout run (colab_launcher_ana_allclass_holdout.ipynb) and its error analysis
RUN_DIR = f"{DRIVE_OUT}/ablation_allclass_holdout_lora_no_extras/runs/A_all_classes_144min/lora_20261026_123804"
ERRORS_CSV = f"{DRIVE_OUT}/allclass_holdout_analysis/errors/errors_lora_all.csv"   # optional: checks the split
PILOT_CAUSES = f"{DRIVE_OUT}/error_causes/causes.json"                             # optional: pilot, for comparison
OUT_NAME = "allclass_holdout_analysis/causes"
# End the runtime after the last cell, or as soon as a step fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])
assert (Path(RUN_DIR) / "last.pt").is_file() and (Path(RUN_DIR) / "config.yaml").is_file(), (
    f"no last.pt / config.yaml in {RUN_DIR}: upload the folder ablation_allclass_holdout_lora_no_extras "
    "to CS604_RUNS (keep its name), or fix RUN_DIR")
print("checkpoint found:", RUN_DIR)

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 5. Rebuild the all-class split, embed every photo and sort the errors into causes

In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

from scripts.pilot_sweep import build_pilot_root

# The same split the all-class run trained on: all 10,000 classes, 5 holdout photos each, seed 0
PILOT_ROOT, info = build_pilot_root(DATA_DIR, PILOT_DATA_DIR, 10000, 5, 0)
print(info)
if Path(ERRORS_CSV).is_file():
    holdout = {line.split()[0] for line in (Path(PILOT_ROOT) / "val.txt").read_text().splitlines() if line.strip()}
    with open(ERRORS_CSV, newline="") as f:
        err_paths = [row["path"] for row in csv.DictReader(f)]
    outside = sum(p not in holdout for p in err_paths)
    assert outside == 0, f"{outside} of {len(err_paths)} error photos are not in the rebuilt holdout: wrong split"
    print(f"split check: all {len(err_paths):,} error photos of the all-class run are in the rebuilt holdout")
else:
    print("no errors file to check the split against; continuing")
OUT = Path(DRIVE_OUT) / OUT_NAME
proc = subprocess.run([sys.executable, "-u", "scripts/error_causes.py", "--run_dir", RUN_DIR,
                       "--pilot_root", str(PILOT_ROOT), "--out_dir", str(OUT)])
if proc.returncode != 0:
    end_runtime(f"error causes failed (exit {proc.returncode})")
    raise SystemExit("error causes failed")

## 6. Results

- **share_of_errors vs same_flag_among_correct_photos:** a cause is a real problem if errors have it far more often than correct photos do.
- **All classes vs pilot:** expect more siblings here (the pilot's 1,000 random classes left out most close relatives).
- **spearman_with_class_accuracy:** negative for messiness or split means messier / two-look classes are less accurate.
- **Galleries:** check that each detector agrees with your eyes.

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary = json.loads((OUT / "causes.json").read_text())
print(json.dumps(summary, indent=2))
table = {"all classes (errors)": summary["share_of_errors"],
         "all classes (correct photos)": summary["same_flag_among_correct_photos"]}
if Path(PILOT_CAUSES).is_file():
    pilot = json.loads(Path(PILOT_CAUSES).read_text())
    table["pilot (errors)"] = pilot["share_of_errors"]
    table["pilot (correct photos)"] = pilot["same_flag_among_correct_photos"]
display(pd.DataFrame(table))
for cause in ("sibling", "unseen_look", "two_looks", "messy_class"):
    png = OUT / f"causes_{cause}.png"
    if png.is_file():
        display(Image(filename=str(png)))

## 7. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")